In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc

# Parameter generation


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

# --- Define sampled parameters ---
gene_params_sampled = [
    "pi_on",            # p_on / (p_on + p_off)
    "p_on",             # directly sampled p_on
    "mrna_half_life",
    "protein_half_life",
    "p_prod_protein",
    "p_prod_mRNA"
]

interaction_params_scaled = [
    "n_gene_1_to_gene_2", "p_add_scaled_gene_1_to_gene_2",
    "n_gene_2_to_gene_1", "p_add_scaled_gene_2_to_gene_1",
    "n_gene_1_to_gene_3", "p_add_scaled_gene_1_to_gene_3",
    "n_gene_2_to_gene_3", "p_add_scaled_gene_2_to_gene_3"
]

param_names = (
    [f"{p}_gene_1" for p in gene_params_sampled] +
    [f"{p}_gene_2" for p in gene_params_sampled] +
    [f"{p}_gene_3" for p in gene_params_sampled] +
    interaction_params_scaled
)

# --- Bounds for sampled parameters ---
param_bounds = {
    # Gene-level
    "pi_on": (0.002, 0.4),                  # activation probability
    "p_on": (0.01, 3),                      # directly sampled
    "mrna_half_life": (0.6, 17),
    "protein_half_life": (7, 200),
    "p_prod_mRNA": (0.2, 60),
    "p_prod_protein": (19, 2700),
    # Interaction-level
    "n_gene_1_to_gene_2": (0.1, 5),
    "n_gene_2_to_gene_1": (0.1, 5),
    "n_gene_1_to_gene_3": (0.1, 5),
    "n_gene_2_to_gene_3": (0.1, 5),
    "p_add_scaled_gene_1_to_gene_2": (0.5, 10),
    "p_add_scaled_gene_2_to_gene_1": (0.5, 10),
    "p_add_scaled_gene_1_to_gene_3": (0.5, 10),
    "p_add_scaled_gene_2_to_gene_3": (0.5, 10),
}

bounds = (
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in interaction_params_scaled]
)

# --- Sampling configuration ---
n_valid_required = 25000
seed = 42

# Latin Hypercube Sampling (log10 for all except pi_on)
log_bounds_lower = [
    np.log10(b[0]) if "pi_on" not in param_names[i] else b[0]
    for i, b in enumerate(bounds)
]
log_bounds_upper = [
    np.log10(b[1]) if "pi_on" not in param_names[i] else b[1]
    for i, b in enumerate(bounds)
]

sampler = qmc.LatinHypercube(d=len(bounds), seed=seed)
sample = sampler.random(n=n_valid_required)

scaled_sample = np.empty_like(sample)
for i, name in enumerate(param_names):
    # Determine log bounds for every parameter (including pi_on)
    lower = np.log10(param_bounds["pi_on"][0]) if "pi_on" in name else log_bounds_lower[i]
    upper = np.log10(param_bounds["pi_on"][1]) if "pi_on" in name else log_bounds_upper[i]

    # Scale in log-space and exponentiate back
    scaled_log = qmc.scale(sample[:, [i]], [lower], [upper]).ravel()
    scaled_sample[:, i] = 10 ** scaled_log



df_sampled = pd.DataFrame(scaled_sample, columns=param_names)

# --- Convert to actual p_off and p_add ---
def convert_params(row):
    converted = {}

    for g in [1, 2, 3]:
        pi = row[f"pi_on_gene_{g}"]
        p_on = row[f"p_on_gene_{g}"]

        # Derive p_off from pi_on and p_on
        p_off = p_on * (1 - pi) / pi

        converted[f"p_on_gene_{g}"] = p_on
        converted[f"p_off_gene_{g}"] = p_off
        converted[f"mrna_half_life_gene_{g}"] = row[f"mrna_half_life_gene_{g}"]
        converted[f"protein_half_life_gene_{g}"] = row[f"protein_half_life_gene_{g}"]
        converted[f"p_prod_mRNA_gene_{g}"] = row[f"p_prod_mRNA_gene_{g}"]
        converted[f"p_prod_protein_gene_{g}"] = row[f"p_prod_protein_gene_{g}"]

    # Convert p_add_scaled → p_add
    for key in row.index:
        if "p_add_scaled" in key:
            tgt_gene = key.split("_")[7]  # target gene index
            p_on_target = converted[f"p_on_gene_{tgt_gene}"]
            converted[key.replace("p_add_scaled", "p_add")] = row[key] * p_on_target

        elif "n_gene" in key:
            converted[key] = row[key]

    return pd.Series(converted)

df_converted = df_sampled.apply(convert_params, axis=1)

# --- Expand to long format ---
rows = []
for idx, row in df_converted.iterrows():
    g1 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_1") and "to" not in k}
    g2 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_2") and "to" not in k}
    g3 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_3") and "to" not in k}
    interactions = {k: v for k, v in row.items() if "gene_" in k and "to" in k}

    rows.append({**g1, **interactions, "pair_id": idx, "gene_id": 1})
    rows.append({**g2, **interactions, "pair_id": idx, "gene_id": 2})
    rows.append({**g3, **interactions, "pair_id": idx, "gene_id": 3})

final_df = pd.DataFrame(rows).reset_index(drop=True)

# --- Save ---
output_path = (
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_positive_reg_pi_on_r_add_scaled.csv"
)
final_df.to_csv(output_path)
print(f"\n✅ Saved to {output_path}")


✅ Saved to /home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_positive_reg_pi_on_r_add_scaled.csv


In [6]:
final_df

,p_on,p_off,mrna_half_life,protein_half_life,p_prod_mRNA,p_prod_protein,pi_on,n_gene_1_to_gene_2,p_add_gene_1_to_gene_2,n_gene_2_to_gene_1,p_add_gene_2_to_gene_1,n_gene_1_to_gene_3,p_add_gene_1_to_gene_3,n_gene_2_to_gene_3,p_add_gene_2_to_gene_3,pair_id,gene_id
0,0.597107,110.976717,13.952608,23.578298,7.316344,698.431153,0.005352,2.027978,3.279028,1.493745,1.356240,0.518471,0.224723,1.804552,0.242889,0,1
1,0.663884,260.019625,0.857989,124.941695,0.483831,958.156197,0.002547,2.027978,3.279028,1.493745,1.356240,0.518471,0.224723,1.804552,0.242889,0,2
2,0.188211,2.319686,5.602431,164.632082,0.583351,84.157268,0.075047,2.027978,3.279028,1.493745,1.356240,0.518471,0.224723,1.804552,0.242889,0,3
3,0.194000,0.540983,3.483315,64.650829,31.458959,505.094901,0.263952,3.685206,0.042474,1.664414,0.658541,0.250028,10.366479,0.525395,7.547116,1,1
4,0.011763,0.220744,1.970430,15.899471,23.569570,127.249441,0.050591,3.685206,0.042474,1.664414,0.658541,0.250028,10.366479,0.525395,7.547116,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74995,2.555842,1262.341691,5.524834,95.664847,10.287376,2304.959783,0.002021,0.657164,22.746573,2.998208,0.124031,0.322139,0.595238,0.155133,0.148383,24998,2
74996,0.090537,10.998567,6.139408,74.573953,4.217145,28.826396,0.008165,0.657164,22.746573,2.998208,0.124031,0.322139,0.595238,0.155133,0.148383,24998,3
74997,0.885408,154.773326,2.671323,15.371391,9.363193,972.452743,0.005688,1.502813,4.438901,0.897876,1.689334,0.417124,0.637610,0.411710,0.651935,24999,1
74998,0.528805,12.092037,1.492842,29.970215,5.442554,156.575730,0.041899,1.502813,4.438901,0.897876,1.689334,0.417124,0.637610,0.411710,0.651935,24999,2


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

# --- Define sampled parameters ---
gene_params_sampled = [
    "pi_on",            # p_on / (p_on + p_off)
    "p_on",             # directly sampled p_on
    "mrna_half_life",
    "protein_half_life",
    "p_prod_protein",
    "p_prod_mRNA"
]

interaction_params_scaled = [
    "n_gene_1_to_gene_2", "p_add_scaled_gene_1_to_gene_2",
    "n_gene_2_to_gene_1", "p_add_scaled_gene_2_to_gene_1",
    "n_gene_1_to_gene_3", "p_add_scaled_gene_1_to_gene_3",
    "n_gene_2_to_gene_3", "p_add_scaled_gene_2_to_gene_3"
]

param_names = (
    [f"{p}_gene_1" for p in gene_params_sampled] +
    [f"{p}_gene_2" for p in gene_params_sampled] +
    [f"{p}_gene_3" for p in gene_params_sampled] +
    interaction_params_scaled
)

# --- Bounds for sampled parameters ---
param_bounds = {
    # Gene-level
    "pi_on": (0.002, 0.4),                  # activation probability
    "p_on": (0.01, 3),                      # directly sampled
    "mrna_half_life": (0.6, 17),
    "protein_half_life": (7, 200),
    "p_prod_mRNA": (0.2, 60),
    "p_prod_protein": (19, 2700),
    # Interaction-level
    "n_gene_1_to_gene_2": (0.1, 5),
    "n_gene_2_to_gene_1": (0.1, 5),
    "n_gene_1_to_gene_3": (0.1, 5),
    "n_gene_2_to_gene_3": (0.1, 5),
    "p_add_scaled_gene_1_to_gene_2": (0.5, 2),
    "p_add_scaled_gene_2_to_gene_1": (0.5, 2),
    "p_add_scaled_gene_1_to_gene_3": (0.5, 2),
    "p_add_scaled_gene_2_to_gene_3": (0.5, 2),
}

bounds = (
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in interaction_params_scaled]
)

# --- Sampling configuration ---
n_valid_required = 25000
seed = 42

# Latin Hypercube Sampling (log10 for all except pi_on)
log_bounds_lower = [
    np.log10(b[0]) if "pi_on" not in param_names[i] else b[0]
    for i, b in enumerate(bounds)
]
log_bounds_upper = [
    np.log10(b[1]) if "pi_on" not in param_names[i] else b[1]
    for i, b in enumerate(bounds)
]

sampler = qmc.LatinHypercube(d=len(bounds), seed=seed)
sample = sampler.random(n=n_valid_required)

scaled_sample = np.empty_like(sample)
for i, name in enumerate(param_names):
    # Determine log bounds for every parameter (including pi_on)
    lower = np.log10(param_bounds["pi_on"][0]) if "pi_on" in name else log_bounds_lower[i]
    upper = np.log10(param_bounds["pi_on"][1]) if "pi_on" in name else log_bounds_upper[i]

    # Scale in log-space and exponentiate back
    scaled_log = qmc.scale(sample[:, [i]], [lower], [upper]).ravel()
    scaled_sample[:, i] = 10 ** scaled_log



df_sampled = pd.DataFrame(scaled_sample, columns=param_names)

# --- Convert to actual p_off and p_add ---
def convert_params(row):
    converted = {}

    for g in [1, 2, 3]:
        pi = row[f"pi_on_gene_{g}"]
        p_on = row[f"p_on_gene_{g}"]

        # Derive p_off from pi_on and p_on
        p_off = p_on * (1 - pi) / pi

        converted[f"p_on_gene_{g}"] = p_on
        converted[f"p_off_gene_{g}"] = p_off
        converted[f"mrna_half_life_gene_{g}"] = row[f"mrna_half_life_gene_{g}"]
        converted[f"protein_half_life_gene_{g}"] = row[f"protein_half_life_gene_{g}"]
        converted[f"p_prod_mRNA_gene_{g}"] = row[f"p_prod_mRNA_gene_{g}"]
        converted[f"p_prod_protein_gene_{g}"] = row[f"p_prod_protein_gene_{g}"]

    # Convert p_add_scaled → p_add
    for key in row.index:
        if "p_add_scaled" in key:
            tgt_gene = key.split("_")[7]  # target gene index
            p_on_target = converted[f"p_on_gene_{tgt_gene}"]
            converted[key.replace("p_add_scaled", "p_add")] = row[key] * p_on_target
        elif "n_gene" in key:
            converted[key] = row[key]

    return pd.Series(converted)

df_converted = df_sampled.apply(convert_params, axis=1)

# --- Expand to long format ---
rows = []
for idx, row in df_converted.iterrows():
    g1 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_1") and "to" not in k}
    g2 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_2") and "to" not in k}
    g3 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_3") and "to" not in k}
    interactions = {k: v for k, v in row.items() if "gene_" in k and "to" in k}

    rows.append({**g1, **interactions, "pair_id": idx, "gene_id": 1})
    rows.append({**g2, **interactions, "pair_id": idx, "gene_id": 2})
    rows.append({**g3, **interactions, "pair_id": idx, "gene_id": 3})

final_df = pd.DataFrame(rows).reset_index(drop=True)

# --- Save ---
output_path = (
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_repression_reg_pi_on_r_add_scaled.csv"
)
final_df.to_csv(output_path)
print(f"\n✅ Saved to {output_path}")


✅ Saved to /home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_repression_reg_pi_on_r_add_scaled.csv


In [1]:
# param_df = pd.read_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation/sim_details/lhc_sampled_parameters_negative_reg.csv", index_col = 0)

import numpy as np
import pandas as pd

def hl_to_deg(hl):
    """Convert half-life to degradation rate."""
    return np.log(2) / hl

def compute_steady_state_levels(param_df, gene_id):
    """Compute mean mRNA and protein levels for gene_id (1 or 2), assuming hill = 0.5."""
    assert gene_id in [1, 2], "gene_id must be 1 or 2"

    # Basic parameters
    p_on = param_df["p_on"]
    p_off = param_df["p_off"]
    prod_m = param_df["p_prod_mRNA"]
    prod_p = param_df["p_prod_protein"]
    deg_m = hl_to_deg(param_df["mrna_half_life"])
    deg_p = hl_to_deg(param_df["protein_half_life"])

    # Use .get to safely retrieve interaction term or default to 0
    if gene_id == 2:
        p_add = param_df.get("p_add_gene_1_to_gene_2", 0.0)
    else:
        p_add = param_df.get("p_add_gene_2_to_gene_1", 0.0)

    # Compute effective p_on using hill response = 0.5
    p_on_eff = p_on + 0.5 * p_add
    burst_prob = p_on_eff / (p_on_eff + p_off)

    # Steady-state means
    mean_mRNA = burst_prob * prod_m / deg_m
    mean_protein = mean_mRNA * prod_p / deg_p

    # Store results in DataFrame
    param_df["mean_mRNA_level"] = mean_mRNA
    param_df["mean_protein_level"] = mean_protein

    return param_df

# Usage
param_df = pd.read_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation_run_2/sim_details/lhc_sampled_parameters_positive_reg_2.csv", index_col = 0)
param_df = compute_steady_state_levels(param_df, gene_id=2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Filter out zero or negative values (log scale can't handle them)
values = param_df['mean_protein_level']
values = values[values > 0]

# Define log-spaced bins
n_bins = 100
min_val = values.min()
max_val = values.max()
log_bins = np.logspace(np.log10(min_val), np.log10(max_val), n_bins)

# Plot
plt.figure(figsize=(6, 4))
plt.hist(values, bins=log_bins)
plt.xscale('log')
# plt.yscale('log')
plt.xlabel("Mean protein level (log scale)")
plt.ylabel("Frequency (log scale)")
plt.title("Log-Binned Histogram of Mean Protein Levels")
plt.tight_layout()
plt.show()


In [ ]:
param_df[param_df['mean_mRNA_level'] < 100].shape

In [ ]:
plt.hist(param_df[param_df['mean_mRNA_level'] < 1]['mean_mRNA_level'])

## Generate test parameters


In [5]:
import pandas as pd
import numpy as np

# Initial values
gene_1 = [0.55, 8.08, 5, 45, 2, 500, 2, 6, 7, 1]
gene_2 = [0.55, 8.08, 5, 45, 2, 500, 2, 6, 7, 2]

r_add = np.arange(2, 10, 0.5)
rows = []

for i, r_add_curr in enumerate(r_add):
    gene_1[7] = r_add_curr
    gene_1[8] = i

    gene_2[7] = r_add_curr
    gene_2[8] = i

    rows.append(gene_1.copy())
    rows.append(gene_2.copy())

gene_1 = [0.55, 1.375, 5, 45, 2, 500, 2, 6, 7, 1]
gene_2 = [0.55, 1.375, 5, 45, 2, 500, 2, 6, 7, 2]

for i, r_add_curr in enumerate(r_add):
    gene_1[7] = r_add_curr
    gene_1[8] = i + 16

    gene_2[7] = r_add_curr
    gene_2[8] = i + 16

    rows.append(gene_1.copy())
    rows.append(gene_2.copy())

gene_1 = [0.55, 275.0, 5, 45, 2, 500, 2, 6, 7, 1]
gene_2 = [0.55, 275.0, 5, 45, 2, 500, 2, 6, 7, 2]

for i, r_add_curr in enumerate(r_add):
    gene_1[7] = r_add_curr
    gene_1[8] = i + 32

    gene_2[7] = r_add_curr
    gene_2[8] = i + 32

    rows.append(gene_1.copy())
    rows.append(gene_2.copy())

# Create DataFrame
df = pd.DataFrame(rows)
df.to_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation_test/sim_details/effect_of_radd_positive_new.csv")



## Creating parameters to sample ranges for testing the impact of $p_{on}$ and $r_{add}$


### $r_{add}$ is sampled independently


In [2]:
import pandas as pd
import numpy as np

# Fixed burst probability
burst_prob = 0.07

# Column headers
columns = [
    "p_on", "p_off", "mrna_half_life", "protein_half_life",
    "p_prod_mRNA", "p_prod_protein", "n_gene_1_to_gene_2",
    "p_add_gene_1_to_gene_2", "pair_id", "gene_id"
]

# Fixed values for gene_1
base_gene_1 = [
    0.55,
    ((1 /burst_prob) - 1)*0.55,
    5, 45, 2, 560, 2, 6,  # p_add placeholder
    0, 1
]

# Base template for gene_2 (will be overwritten)
base_gene_2 = [
    0.0, 0.0, 5, 45, 2, 560, 2, 0.0,  # r_add will be updated
    0, 2
]

# Grid sampling
k_on_values = np.logspace(np.log10(0.01), np.log10(3.0), 15)
r_add_values = np.linspace(2.0, 10.0, 9)

rows = []
pair_id = 0

for p_on in k_on_values:
    p_off =((1 /burst_prob) - 1)*p_on
    for r_add in r_add_values:
        # Gene 1 (fixed, except pair_id)
        gene_1 = base_gene_1.copy()
        gene_1[7] = r_add
        gene_1[8] = pair_id  # pair_id

        # Gene 2 (grid-sampled)
        gene_2 = base_gene_2.copy()
        gene_2[0] = p_on
        gene_2[1] = p_off
        gene_2[7] = r_add
        gene_2[8] = pair_id  # pair_id

        rows.append(gene_1)
        rows.append(gene_2)

        pair_id += 1

# Convert and save
df = pd.DataFrame(rows, columns=columns)
df.to_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation_test/sim_details/effect_of_r_add_sampling_independent.csv")


In [23]:
df

,p_on,p_off,mrna_half_life,protein_half_life,p_prod_mRNA,p_prod_protein,n_gene_1_to_gene_2,p_add_gene_1_to_gene_2,pair_id,gene_id
0,0.55,7.307143,5,45,2,560,2,2.0,0,1
1,0.01,0.132857,5,45,2,560,2,2.0,0,2
2,0.55,7.307143,5,45,2,560,2,3.0,1,1
3,0.01,0.132857,5,45,2,560,2,3.0,1,2
4,0.55,7.307143,5,45,2,560,2,4.0,2,1
...,...,...,...,...,...,...,...,...,...,...
265,3.00,39.857143,5,45,2,560,2,8.0,132,2
266,0.55,7.307143,5,45,2,560,2,9.0,133,1
267,3.00,39.857143,5,45,2,560,2,9.0,133,2
268,0.55,7.307143,5,45,2,560,2,10.0,134,1


### $r_{add}$ is sampled as a ratio of $k_{off}$


In [1]:
import pandas as pd
import numpy as np

# Fixed burst probability
burst_prob = 0.07

# Column headers
columns = [
    "p_on", "p_off", "mrna_half_life", "protein_half_life",
    "p_prod_mRNA", "p_prod_protein", "n_gene_1_to_gene_2",
    "p_add_gene_1_to_gene_2", "pair_id", "gene_id"
]

# Fixed values for gene_1
base_gene_1 = [
    0.55,
    ((1 /burst_prob) - 1)*0.55,
    5, 45, 2, 560, 2, 6,  # p_add placeholder
    0, 1
]

# Base template for gene_2 (will be overwritten)
base_gene_2 = [
    0.0, 0.0, 5, 45, 2, 560, 2, 0.0,  # r_add will be updated
    0, 2
]

# Grid sampling
k_on_values = np.logspace(np.log10(0.01), np.log10(3.0), 15)
r_add_ratios = np.linspace(0.25, 1.5, 9)

rows = []
pair_id = 0

for p_on in k_on_values:
    p_off =((1 /burst_prob) - 1)*p_on
    for r_add_ratio in r_add_ratios:
        # Gene 1 (fixed, except pair_id)
        gene_1 = base_gene_1.copy()
        gene_1[8] = pair_id  # pair_id

        # Gene 2 (grid-sampled)
        gene_2 = base_gene_2.copy()
        gene_2[0] = p_on
        gene_2[1] = p_off
        gene_1[7] = r_add_ratio*p_off
        gene_2[7] = r_add_ratio*p_off
        gene_2[8] = pair_id  # pair_id

        rows.append(gene_1)
        rows.append(gene_2)

        pair_id += 1

# Convert and save
df = pd.DataFrame(rows, columns=columns)
df.to_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation_test/sim_details/effect_of_r_add_sampling_dependent.csv")


In [25]:
df

,p_on,p_off,mrna_half_life,protein_half_life,p_prod_mRNA,p_prod_protein,n_gene_1_to_gene_2,p_add_gene_1_to_gene_2,pair_id,gene_id
0,0.55,7.307143,5,45,2,560,2,0.033214,0,1
1,0.01,0.132857,5,45,2,560,2,0.033214,0,2
2,0.55,7.307143,5,45,2,560,2,0.053973,1,1
3,0.01,0.132857,5,45,2,560,2,0.053973,1,2
4,0.55,7.307143,5,45,2,560,2,0.074732,2,1
...,...,...,...,...,...,...,...,...,...,...
265,3.00,39.857143,5,45,2,560,2,47.330357,132,2
266,0.55,7.307143,5,45,2,560,2,53.558036,133,1
267,3.00,39.857143,5,45,2,560,2,53.558036,133,2
268,0.55,7.307143,5,45,2,560,2,59.785714,134,1
